In [1]:
import os
from pathlib import Path

from huggingface_hub import login
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

In [23]:
PDF_PATH = "../data/gym-policy-terms.pdf"

In [24]:
docs = PyPDFLoader(PDF_PATH).load()

In [34]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, 
    chunk_overlap=300,
    separators = ["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(docs)

In [37]:
print(len(chunks))
print(len(chunks[0].page_content))
print(chunks[0], "\n\n", chunks[1], "\n\n", chunks[2])

20
1166
page_content='Chính Sách & Điều Khoản Sử Dụng Phòng 
Gym 
 
1. Giới thiệu 
1. "Phòng Gym" hoặc "Chúng tôi" là đơn vị vận hành trung tâm thể dục thể thao cung cấp 
dịch vụ tập luyện, tiện ích đi kèm và các dịch vụ giá trị gia tăng (ví dụ: lớp nhóm, huấn 
luyện viên cá nhân, spa,…). 
2. "Hội viên" hoặc "Bạn" là cá nhân đăng ký sử dụng dịch vụ của Phòng Gym thông qua 
hợp đồng, phiếu đăng ký, ứng dụng hoặc website. 
3. Tài liệu này quy định quyền lợi, nghĩa vụ và trách nhiệm của Hội viên và Phòng Gym 
liên quan đến việc sử dụng dịch vụ, bao gồm: quyền truy cập các khu vực, quy định an 
toàn, bảo mật thông tin cá nhân, thanh toán, hoàn/huỷ, kỷ luật và chấm dứt. 
4. Khi đăng ký hội viên, sử dụng thẻ/QR vào cổng, đặt lớp, sử dụng thiết bị hoặc truy cập hệ 
thống online của Phòng Gym, bạn được xem là đã đọc, hiểu và đồng ý bị ràng buộc bởi 
các điều khoản dưới đây. 
5. Phòng Gym hoạt động 24/7, tất cả các ngày trong năm kể cả ngày lễ và Tết. Một số khu 
vực có thể có giờ hoạt động riê

In [38]:
print(chunks[2])

page_content='1. Hội viên phải từ đủ 16 tuổi. Người dưới 18 tuổi cần có sự đồng ý bằng văn bản của 
cha/mẹ hoặc người giám hộ hợp pháp. 
2. Hội viên cam kết có đủ sức khoẻ cơ bản để tham gia tập luyện. Phòng Gym khuyến nghị 
bạn tham khảo ý kiến bác sĩ nếu có bệnh nền (tim mạch, huyết áp, xương khớp, thai 
kỳ,…). 
3. Phòng Gym có quyền từ chối hoặc chấm dứt hội viên nếu: 
o Cung cấp thông tin giả mạo, không chính xác, hoặc dùng giấy tờ của người khác. 
o Có hành vi vi phạm nội quy, gây rối trật tự, đe doạ an toàn của người khác. 
o Không tuân thủ chỉ dẫn an toàn của nhân viên. 
 
4. Đăng ký và xác thực thông tin 
1. Khi đăng ký, Hội viên cần cung cấp tối thiểu: họ tên, ngày sinh, số điện thoại, email (nếu 
có), ảnh chân dung hoặc thông tin nhận diện tương đương. 
2. Phòng Gym có thể yêu cầu xuất trình giấy tờ tuỳ thân (CCCD, CMND, hộ chiếu, thẻ học 
sinh/sinh viên) để xác thực. 
3. Hội viên chịu trách nhiệm cập nhật thông tin liên hệ (số điện thoại, email, địa chỉ) khi có 
thay đổi. Cá

In [39]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    encode_kwargs={"normalize_embeddings": True}
)

In [40]:
COLLECTION_NAME = "gym-policy-terms"
PERSIST_DIR = "../data/policy-terms-db"

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME
)
